In [4]:
import pandas as pd
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. 데이터 로드
# =============================================
print("=== 데이터 로드 ===")

games = pd.read_csv("games.csv")
recs = pd.read_csv("recommendations.csv")

with open("games_metadata.json", "r") as f:
    meta_list = [json.loads(line) for line in f if line.strip()]
meta_a = pd.DataFrame(meta_list)

games_c = pd.read_csv("games_march2025_cleaned.csv")

# =============================================
# 2. A + C 병합
# =============================================
print("\n=== A + C 병합 ===")

if 'appid' in games_c.columns:
    games_c = games_c.rename(columns={'appid': 'app_id'})

c_cols = ['app_id']
for col in ['tags', 'genres', 'about_the_game', 'detailed_description', 'categories']:
    if col in games_c.columns:
        c_cols.append(col)

games_merged = games.merge(games_c[c_cols], on='app_id', how='left')
print(f"병합 완료: {games_merged.shape}")

# =============================================
# 3. 연도 기준 분할
# =============================================
print("\n=== 연도 기준 분할 ===")

recs['date'] = pd.to_datetime(recs['date'])
SPLIT_YEAR = '2022-01-01'

train_recs = recs[recs['date'] < SPLIT_YEAR]
test_recs = recs[recs['date'] >= SPLIT_YEAR]

print(f"Train: {len(train_recs):,} / Test: {len(test_recs):,}")

# =============================================
# 4. 전처리
# =============================================
print("\n=== 전처리 ===")

linux_app_ids = set(games[games['linux'] == True]['app_id'])

train_pos = train_recs[
    (train_recs['is_recommended'] == True) &
    (train_recs['app_id'].isin(linux_app_ids))
]
test_pos = test_recs[
    (test_recs['is_recommended'] == True) &
    (test_recs['app_id'].isin(linux_app_ids))
]

user_counts = train_pos['user_id'].value_counts()
valid_users = user_counts[user_counts >= 5].index
train_pos = train_pos[train_pos['user_id'].isin(valid_users)]

print(f"Train 긍정: {len(train_pos):,}")
print(f"Test 긍정: {len(test_pos):,}")

# =============================================
# 5. 콘텐츠 기반 (TF-IDF)
# =============================================
print("\n=== TF-IDF ===")

games_merged['content'] = ''
for col in ['about_the_game', 'detailed_description', 'tags', 'genres', 'categories']:
    if col in games_merged.columns:
        games_merged['content'] += games_merged[col].fillna('').astype(str) + ' '

linux_content = games_merged[games_merged['app_id'].isin(linux_app_ids)].reset_index(drop=True)

tfidf = TfidfVectorizer(stop_words='english', max_features=5000, min_df=2)
tfidf_matrix = tfidf.fit_transform(linux_content['content'])
cosine_sim = cosine_similarity(tfidf_matrix)

app_to_idx = {app: i for i, app in enumerate(linux_content['app_id'])}

def content_recommend_user(user_games, top_n=10):
    scores = np.zeros(len(linux_content))
    valid_count = 0

    for app_id in user_games[:10]:
        if app_id not in app_to_idx:
            continue
        idx = app_to_idx[app_id]
        scores += cosine_sim[idx]
        valid_count += 1

    if valid_count == 0:
        return []

    scores /= valid_count

    for app_id in user_games:
        if app_id in app_to_idx:
            scores[app_to_idx[app_id]] = -1

    top_indices = scores.argsort()[::-1][:top_n]
    return [linux_content.iloc[i]['app_id'] for i in top_indices]

# =============================================
# 6. 협업 필터링 (Item-KNN)
# =============================================
print("=== Item-KNN ===")

user_item = train_pos.pivot_table(
    index='user_id', columns='app_id',
    values='is_recommended', fill_value=0
)
sparse = csr_matrix(user_item.values)

knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=11)
knn.fit(sparse.T)

def knn_recommend(app_id, top_n=10):
    if app_id not in user_item.columns:
        return []
    idx = user_item.columns.get_loc(app_id)
    _, indices = knn.kneighbors(sparse.T[idx], n_neighbors=top_n+1)
    return [user_item.columns[i] for i in indices.flatten()[1:]]

# =============================================
# 7. 평가 함수
# =============================================
def recall_at_k(rec, actual, k=10):
    if len(actual) == 0:
        return 0.0
    return len(set(rec[:k]) & set(actual)) / min(len(actual), k)

def ndcg_at_k(rec, actual, k=10):
    if len(actual) == 0:
        return 0.0
    actual_set = set(actual)
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(rec[:k]) if item in actual_set)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(actual), k)))
    return dcg / idcg if idcg > 0 else 0.0

def diversity(recommendations_list):
    """추천 다양성: 유저 간 추천 겹침이 적을수록 높음"""
    if len(recommendations_list) < 2:
        return 0.0
    total_overlap = 0
    count = 0
    for i in range(len(recommendations_list)):
        for j in range(i + 1, len(recommendations_list)):
            set_i = set(recommendations_list[i])
            set_j = set(recommendations_list[j])
            if len(set_i) > 0 and len(set_j) > 0:
                overlap = len(set_i & set_j) / len(set_i | set_j)
                total_overlap += overlap
                count += 1
    return 1 - (total_overlap / count) if count > 0 else 0.0

def coverage(all_recommended, total_items):
    """커버리지: 전체 아이템 중 추천된 비율"""
    return len(all_recommended) / total_items if total_items > 0 else 0.0

# =============================================
# 8. 평가 실행
# =============================================
print("\n=== 평가 ===")

sample_users = test_pos['user_id'].unique()[:200]
c_recall, c_ndcg, k_recall, k_ndcg = [], [], [], []
c_all_recs, k_all_recs = [], []
c_recommended_set, k_recommended_set = set(), set()

for uid in sample_users:
    train_games = train_pos[train_pos['user_id'] == uid]['app_id'].values
    test_games = test_pos[test_pos['user_id'] == uid]['app_id'].values
    if len(train_games) == 0:
        continue

    c_recs = content_recommend_user(train_games)
    k_recs = knn_recommend(train_games[0])

    c_recall.append(recall_at_k(c_recs, test_games))
    c_ndcg.append(ndcg_at_k(c_recs, test_games))
    k_recall.append(recall_at_k(k_recs, test_games))
    k_ndcg.append(ndcg_at_k(k_recs, test_games))

    c_all_recs.append(c_recs)
    k_all_recs.append(k_recs)
    c_recommended_set.update(c_recs)
    k_recommended_set.update(k_recs)

# 결과 출력
print(f"\n[콘텐츠 기반]")
print(f"  Recall@10: {np.mean(c_recall):.4f}")
print(f"  NDCG@10: {np.mean(c_ndcg):.4f}")
print(f"  Diversity: {diversity(c_all_recs):.4f}")
print(f"  Coverage: {coverage(c_recommended_set, len(linux_app_ids)):.4f}")

print(f"\n[협업 필터링]")
print(f"  Recall@10: {np.mean(k_recall):.4f}")
print(f"  NDCG@10: {np.mean(k_ndcg):.4f}")
print(f"  Diversity: {diversity(k_all_recs):.4f}")
print(f"  Coverage: {coverage(k_recommended_set, len(linux_app_ids)):.4f}")

print("\n=== 완료 ===")

=== 데이터 로드 ===

=== A + C 병합 ===
병합 완료: (50872, 18)

=== 연도 기준 분할 ===
Train: 31,565,267 / Test: 9,589,527

=== 전처리 ===
Train 긍정: 2,474,413
Test 긍정: 1,813,755

=== TF-IDF ===
=== Item-KNN ===

=== 평가 ===

[콘텐츠 기반]
  Recall@10: 0.0143
  NDCG@10: 0.0135
  Diversity: 0.9325
  Coverage: 0.0059

[협업 필터링]
  Recall@10: 0.3561
  NDCG@10: 0.2037
  Diversity: 0.8988
  Coverage: 0.0051

=== 완료 ===
